# 02 — NLP Basics: Text to Numbers

> **Goal:** Understand how computers process human language.

Computers only understand numbers. But language is text.
NLP answers one question: **how do we turn text into numbers that carry meaning?**

```
Raw text  →  Tokenize  →  Encode  →  Embed  →  Neural Network
'I love AI'  ['I','love','AI']  [4,7,0]  [[0.2,-0.1,...]]  ...
```

In [1]:
import numpy as np
print('Libraries laoded ✓')

Libraries laoded ✓


---
## Part 1 — Tokenization

**Tokenization** = split text into small pieces called **tokens**.

Simplest approach: one word = one token.


In [2]:
# ── Step 1: Tokenization ─────────────────────────────────

def tokenize(text):
    """
    Convert raw text into a list of tokens.
    Steps:
      1. Lowercase everything  → 'Love' and 'love' are the same word
      2. Remove punctuation    → 'learning,' and 'learning' are the same
      3. Split by space        → each word becomes a token
    """
    text = text.lower()
    text = text.replace(',', '').replace('.', '').replace('!', '').replace('?', '')
    tokens = text.split(' ')
    return tokens



In [3]:
# example
sentences = [
    'I love deep learning',
    'Deep learning is amazing',
    'I love AI',
    'AI is the future',
]

print('Tokenization results:')
print('-' * 50)
for s in sentences:
    tokens = tokenize(s)
    print(f"  '{s}'")
    print(f"   → {tokens}")
    print()

Tokenization results:
--------------------------------------------------
  'I love deep learning'
   → ['i', 'love', 'deep', 'learning']

  'Deep learning is amazing'
   → ['deep', 'learning', 'is', 'amazing']

  'I love AI'
   → ['i', 'love', 'ai']

  'AI is the future'
   → ['ai', 'is', 'the', 'future']



---
## Part 2 — Vocabulary & Encoding

After tokenizing, we build a **vocabulary** — a dictionary that maps every unique word to a number.

```
'ai'       → 0
'amazing'  → 1
'deep'     → 2
...
```

In [4]:
# ── Step 2: Build Vocabulary ──────────────────────────────

# Collect all tokens from all sentences
all_tokens = []
for s in sentences:
    all_tokens.extend(tokenize(s))

print(f'All tokens (with duplicates): {all_tokens}')

# Remove duplicates and sort
vocabulary = sorted(set(all_tokens))
print(f'\nVocabulary (unique only): {vocabulary}')
print(f'Vocabulary size: {len(vocabulary)}')

# Create word → index mapping
word_to_idx = {word: idx for idx, word in enumerate(vocabulary)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

print(f'\nword_to_idx:')
for word, idx in word_to_idx.items():
    print(f"  '{word}' → {idx}")

All tokens (with duplicates): ['i', 'love', 'deep', 'learning', 'deep', 'learning', 'is', 'amazing', 'i', 'love', 'ai', 'ai', 'is', 'the', 'future']

Vocabulary (unique only): ['ai', 'amazing', 'deep', 'future', 'i', 'is', 'learning', 'love', 'the']
Vocabulary size: 9

word_to_idx:
  'ai' → 0
  'amazing' → 1
  'deep' → 2
  'future' → 3
  'i' → 4
  'is' → 5
  'learning' → 6
  'love' → 7
  'the' → 8


In [5]:
# ── Step 3: Encoding ──────────────────────────────────────
# Convert a sentence into a list of numbers

def encode(sentence):
    """Text → list of numbers using vocabulary"""
    tokens = tokenize(sentence)
    return [word_to_idx[t] for t in tokens]

def decode(indices):
    """List of numbers → text using vocabulary"""
    return ' '.join([idx_to_word[i] for i in indices])


print('Encoding sentences:')
print('-' * 50)
for s in sentences:
    encoded = encode(s)
    decoded = decode(encoded)
    print(f"  original: '{s}'")
    print(f"  encoded:  {encoded}")
    print(f"  decoded:  '{decoded}'")
    print(f"  round-trip correct: {s.lower() == decoded}")
    print()

Encoding sentences:
--------------------------------------------------
  original: 'I love deep learning'
  encoded:  [4, 7, 2, 6]
  decoded:  'i love deep learning'
  round-trip correct: True

  original: 'Deep learning is amazing'
  encoded:  [2, 6, 5, 1]
  decoded:  'deep learning is amazing'
  round-trip correct: True

  original: 'I love AI'
  encoded:  [4, 7, 0]
  decoded:  'i love ai'
  round-trip correct: True

  original: 'AI is the future'
  encoded:  [0, 5, 8, 3]
  decoded:  'ai is the future'
  round-trip correct: True



---
## Part 3 — One-Hot Encoding (Old Way)

The simplest way to represent a word as a vector:
- All zeros, except one position which is 1

```
vocab = ['ai', 'amazing', 'deep', 'future', 'i', 'is', 'learning', 'love', 'the']

'love' (index=7) → [0, 0, 0, 0, 0, 0, 0, 1, 0]
```

**Problem:** All words are equally distant from each other — no meaning!


In [8]:
# ── One-Hot Encoding ──────────────────────────────────────

vocab_size = len(vocabulary)

def one_hot(word):
    """Word → vector with a single 1, rest zeros"""
    idx = word_to_idx[word]
    vector = np.zeros(vocab_size)   # all zeros
    vector[idx] = 1                  # one position = 1
    return vector


# Show one-hot vectors
words_to_show = ['i', 'love', 'deep', 'ai']
print('One-hot vectors:')
print(f"{'Word':<10} {'Vector'}")
print('-' * 55)
for w in words_to_show:
    vec = one_hot(w).astype(int)
    print(f"  '{w}'{'':.<7} {vec}")

# Show the problem: all distances are equal
print('\nProblem — all distances are equal:')
pairs = [('love', 'amazing'), ('love', 'deep'), ('i', 'ai')]
for w1, w2 in pairs:
    dist = np.linalg.norm(one_hot(w1) - one_hot(w2))
    print(f"  distance('{w1}', '{w2}') = {dist:.2f}")
print("\n→ All 1.41! Computer can't tell which words are related.")

One-hot vectors:
Word       Vector
-------------------------------------------------------
  'i'....... [0 0 0 0 1 0 0 0 0]
  'love'....... [0 0 0 0 0 0 0 1 0]
  'deep'....... [0 0 1 0 0 0 0 0 0]
  'ai'....... [1 0 0 0 0 0 0 0 0]

Problem — all distances are equal:
  distance('love', 'amazing') = 1.41
  distance('love', 'deep') = 1.41
  distance('i', 'ai') = 1.41

→ All 1.41! Computer can't tell which words are related.


---
## Part 4 — Word Embeddings (Modern Way) ⭐

Instead of one-hot, each word gets a **dense vector** of real numbers.
These numbers are **learned** during training — similar words get similar vectors.

```
'king'   → [ 0.20,  0.90,  0.10]  ─┐ close together
'queen'  → [ 0.10,  0.80,  0.90]  ─┘
'apple'  → [ 0.90, -0.50,  0.10]  ← far away
```

Famous example: **king − man + woman ≈ queen**



In [9]:
# ── Word Embeddings ───────────────────────────────────────

embedding_dim = 3   # each word → 3 numbers
                    # in GPT2: each word → 768 numbers
                    # in GPT4: each word → 12288 numbers

# Embedding table: random at start, learned during training
np.random.seed(42)
embedding_table = np.random.randn(vocab_size, embedding_dim)

def get_embedding(word):
    """Look up embedding for a word — just get the right row"""
    idx = word_to_idx[word]
    return embedding_table[idx]   # row lookup


print(f'Embedding table shape: {embedding_table.shape}')
print(f'({vocab_size} words × {embedding_dim} numbers per word)\n')

print('Each word has a vector:')
for word in vocabulary:
    vec = get_embedding(word)
    print(f"  '{word}' → {np.round(vec, 2)}")

Embedding table shape: (9, 3)
(9 words × 3 numbers per word)

Each word has a vector:
  'ai' → [ 0.5  -0.14  0.65]
  'amazing' → [ 1.52 -0.23 -0.23]
  'deep' → [ 1.58  0.77 -0.47]
  'future' → [ 0.54 -0.46 -0.47]
  'i' → [ 0.24 -1.91 -1.72]
  'is' → [-0.56 -1.01  0.31]
  'learning' → [-0.91 -1.41  1.47]
  'love' → [-0.23  0.07 -1.42]
  'the' → [-0.54  0.11 -1.15]


In [10]:
# ── Full Pipeline: Sentence → Embeddings ─────────────────

def sentence_to_embeddings(sentence):
    """Convert a full sentence to a matrix of embeddings"""
    tokens  = tokenize(sentence)
    indices = encode(sentence)
    embeddings = [get_embedding(t) for t in tokens]
    return tokens, indices, np.array(embeddings)


test_sentence = 'I love AI'
tokens, indices, embeddings = sentence_to_embeddings(test_sentence)

print(f'Full pipeline for: "{test_sentence}"')
print(f'\nStep 1 — Tokenize:  {tokens}')
print(f'Step 2 — Encode:    {indices}')
print(f'Step 3 — Embed:')
for token, idx, emb in zip(tokens, indices, embeddings):
    print(f"  {idx} ('{token}') → {np.round(emb, 2)}")
print(f'\nFinal matrix shape: {embeddings.shape}')
print(f'({len(tokens)} words × {embedding_dim} numbers)')
print('\n→ This matrix is the input to the neural network!')


Full pipeline for: "I love AI"

Step 1 — Tokenize:  ['i', 'love', 'ai']
Step 2 — Encode:    [4, 7, 0]
Step 3 — Embed:
  4 ('i') → [ 0.24 -1.91 -1.72]
  7 ('love') → [-0.23  0.07 -1.42]
  0 ('ai') → [ 0.5  -0.14  0.65]

Final matrix shape: (3, 3)
(3 words × 3 numbers)

→ This matrix is the input to the neural network!


---
## Part 5 — One-Hot vs Embedding

| | One-Hot | Embedding |
|---|---|---|
| Size per word | vocab_size numbers | embedding_dim numbers |
| Carries meaning? | ✗ No | ✓ Yes |
| Similar words close? | ✗ No | ✓ Yes |
| Learned? | ✗ Fixed | ✓ Learned during training |
| Used in GPT? | ✗ No | ✓ Yes |

In [11]:
# ── Comparison ────────────────────────────────────────────

print('One-Hot vs Embedding comparison:')
print('-' * 50)
print(f"{'Method':<15} {'Size per word':<20} {'Meaning?'}")
print(f"  {'One-Hot':<13} {vocab_size} numbers{'':8} No")
print(f"  {'Embedding':<13} {embedding_dim} numbers{'':11} Yes (learned)")
print()
print(f'In GPT-4: each word → 12,288 numbers')
print(f'In our code: each word → {embedding_dim} numbers (for simplicity)')

One-Hot vs Embedding comparison:
--------------------------------------------------
Method          Size per word        Meaning?
  One-Hot       9 numbers         No
  Embedding     3 numbers            Yes (learned)

In GPT-4: each word → 12,288 numbers
In our code: each word → 3 numbers (for simplicity)


---
## Summary

It is covered the full pipeline from raw text to numbers ready for a neural network:
1. **Tokenize** — split text into words
2. **Vocabulary** — assign a number to each unique word
3. **One-Hot** — simple but carries no meaning
4. **Embedding** — dense vectors that carry semantic meaning, learned during training

---
**Connection to LLMs:**
When you type a message to GPT or Claude, the first thing that happens is exactly this pipeline — your text gets tokenized and converted to embeddings before the model processes it.
---
**Next:** `03_attention.ipynb` — how the model decides which words to focus on